# Data collection

In [ ]:
import os
import requests
import time
import pandas as pd
from dotenv import load_dotenv
import time

# Load environment variables from the .env file
load_dotenv()

# Grab the key securely
API_KEY = os.getenv("RIOT_API_KEY")

if not API_KEY:
    print("WARNING: API Key not found. Check your .env file!")

headers = {
    "X-Riot-Token": API_KEY
}

# Riot API uses two different routing systems depending on the endpoint.
# Platform routing (na1, euw1) is used for ladders and summoner profiles.
# Regional routing (americas, europe) is used for match histories.
PLATFORM = "euW1" 
REGION = "europe"

## Ratelimit helper

In [ ]:
def make_api_request(url):

    #A wrapper for requests.get() that automatically handles standard Riot API rate limits.
    while True:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            return response.json()
            
        elif response.status_code == 429: # Too Many Requests 
            retry_after = int(response.headers.get("Retry-After", 10)) 
            print(f"Rate limit hit. Sleeping for {retry_after} seconds...")
            time.sleep(retry_after)
            
        elif response.status_code in [403, 401]:
            print("API Key expired or invalid. Please check your Riot Developer portal.")
            return None
            
        else:
            print(f"Error {response.status_code} for URL: {url}")
            return None

## Challenger ladder

In [ ]:
def get_challenger_puuids(limit=10):
    #Pulls the top Ranked Solo Challenger players and retrieves their PUUIDs.

    print("Fetching Challenger Ladder...")
    ladder_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
    ladder_data = make_api_request(ladder_url)
    
    if not ladder_data:
        return []

    # Get the top X players from the ladder entries
    entries = ladder_data.get('entries', [])[:limit]
    
    puuids = []
    print(f"Extracting PUUIDs for the top {limit} players...")
    
    for entry in entries:
        # Riot's newer API data structures use PUUID
        if 'puuid' in entry:
            puuids.append(entry['puuid'])
            
        # Fallback just in case some legacy data still relies on summonerId
        elif 'summonerId' in entry:
            summoner_id = entry['summonerId']
            sum_url = f"https://{PLATFORM}.api.riotgames.com/lol/summoner/v4/summoners/{summoner_id}"
            sum_data = make_api_request(sum_url)
            
            if sum_data and 'puuid' in sum_data:
                puuids.append(sum_data['puuid'])
                
            time.sleep(1.2) 
        
    return puuids

# Test the ladder approach
ladder_seed_puuids = get_challenger_puuids(limit=50)
print(f"Successfully grabbed {len(ladder_seed_puuids)} PUUIDs from the ladder.")

print("Discovered PUUIDs:")
for p in list(ladder_seed_puuids):
    print(p) 

## Snowball Method

In [ ]:
def snowball_players_from_seed(seed_puuid, match_count=5):

    #Takes a single player, gets their recent matches, and extracts all other players from those matches.
    # queue=420 restricts the search to Ranked Solo/Duo games
    
    matches_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{seed_puuid}/ids?queue=420&start=0&count={match_count}"
    match_ids = make_api_request(matches_url)
    
    if not match_ids:
        print("No matches found or error occurred.")
        return set()

    discovered_puuids = set() # Using a set prevents duplicate players
    
    print(f"Found {len(match_ids)} matches. Scraping other players...")
    
    for match_id in match_ids:
        match_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
        match_data = make_api_request(match_url)
        
        if match_data and 'metadata' in match_data:
            # The metadata block contains a clean list of all 10 participants' PUUIDs
            participants = match_data['metadata']['participants']
            for p in participants:
                discovered_puuids.add(p)
                
        time.sleep(1.2) # Rate limit safety

    # Remove the seed player from the list of newly discovered ones (optional)
    if seed_puuid in discovered_puuids:
        discovered_puuids.remove(seed_puuid)
        
    return discovered_puuids

# Test the snowball approach using the first PUUID we got from the ladder above
if ladder_seed_puuids:
    seed = ladder_seed_puuids[0]
    print(f"Snowballing from seed PUUID: {seed[:15]}...")
    new_players = snowball_players_from_seed(seed, match_count=3)
    print(f"Discovered {len(new_players)} new players to analyze!")

    print("Discovered PUUIDs:")
    for p in list(new_players):
        print(p) 

## Get match history and parse the data

In [ ]:
def get_player_match_history(puuid, total_matches=250):
    """
    Fetches a large number of Match IDs for a player by paginating 
    through the API 100 matches at a time.
    """
    all_match_ids = []
    start_index = 0
    
    print(f"Fetching up to {total_matches} match IDs for player...")
    
    while len(all_match_ids) < total_matches:
        # Calculate how many matches to ask for in this specific request
        # (It will be 100, unless we only need a few more to hit our total)
        matches_left = total_matches - len(all_match_ids)
        current_count = min(100, matches_left)
        
        # queue=420 filters for Ranked Solo/Duo
        url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?queue=420&start={start_index}&count={current_count}"
        
        print(f" -> Requesting {current_count} matches starting at index {start_index}...")
        batch_ids = make_api_request(url)
        
        # If the API returns an empty list, we've reached the end of their match history!
        if not batch_ids:
            print("Reached the end of the player's match history.")
            break
            
        all_match_ids.extend(batch_ids)
        start_index += current_count
        
        # Respect the rate limit between pagination requests
        time.sleep(1.2)
        
    print(f"Successfully grabbed {len(all_match_ids)} total match IDs.")
    return all_match_ids

In [ ]:
def parse_match_data(match_id, target_puuid):
    """
    Pulls the detailed stats for a match and extracts the specific 
    variables needed for session analysis.
    """
    url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    match_data = make_api_request(url)
    
    if not match_data or 'info' not in match_data:
        return None

    info = match_data['info']
    
    # Find our target player among the 10 participants
    participants = info.get('participants', [])
    player_data = next((p for p in participants if p['puuid'] == target_puuid), None)
    
    if not player_data:
        return None

    # Extract the core data points for your project
    parsed_data = {
        "match_id": match_id,
        "puuid": target_puuid,
        "game_start_timestamp": info.get("gameStartTimestamp"), # Milliseconds since epoch
        "game_end_timestamp": info.get("gameEndTimestamp"),     # Milliseconds since epoch
        "game_duration_sec": info.get("gameDuration"),          # Seconds
        "win": player_data.get("win"),                          # Boolean (True/False)
        "champion_played": player_data.get("championName"),
        "kills": player_data.get("kills"),
        "deaths": player_data.get("deaths"),
        "assists": player_data.get("assists")
    }
    
    return parsed_data

## Player Rank

In [ ]:
def get_player_rank(puuid):
    """
    Fetches a player's Ranked Solo/Duo rank directly using their PUUID.
    Returns a string like 'GOLD II' or 'UNRANKED'.
    """
    league_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}"
    league_data = make_api_request(league_url)
    
    if league_data is None:
        return "API_ERROR"
        
    if not league_data: # If the list is empty, they haven't played ranked
        return "UNRANKED"
        
    # A player can have multiple ranks (Solo/Duo, Flex). We only want Solo/Duo.
    for queue in league_data:
        if queue.get('queueType') == 'RANKED_SOLO_5x5':
            tier = queue.get('tier', '')
            rank = queue.get('rank', '')
            return f"{tier} {rank}" # e.g., "DIAMOND IV"
            
    return "UNRANKED (Solo/Duo)"

## Save data

In [ ]:
def save_player_data(player_df, puuid):
    """
    Saves a single player's DataFrame to the /data folder.
    Using the PUUID in the filename ensures we don't overwrite files.
    """
    if player_df is not None and not player_df.empty:
        # Create a safe filename using the first 15 characters of the PUUID
        filename = f"player_{puuid[:15]}_matches.csv"
        file_path = os.path.join("Data/", filename)
        
        # Save to CSV without the index column
        player_df.to_csv(file_path, index=False)
        print(f"Saved {len(player_df)} matches to {file_path}")
    else:
        print(f"No data to save for player {puuid[:15]}")

## Test

In [ ]:
ladder_seed_puuids = get_challenger_puuids(limit=20)
print(f"Successfully grabbed {len(ladder_seed_puuids)} PUUIDs from the Challenger ladder.")

# Convert to list and get total for cleaner iteration
players_list = list(ladder_seed_puuids)
total_players = len(players_list)

# --- NEW: Start a timer for the entire script ---
script_start_time = time.time()

for player_index, p in enumerate(players_list):
    target_player = p
    
    # --- NEW: Calculate Global ETA based on fully completed players ---
    if player_index > 0:
        script_elapsed = time.time() - script_start_time
        avg_time_per_player = script_elapsed / player_index
        players_left = total_players - player_index
        global_eta_seconds = avg_time_per_player * players_left
        
        # Format into Hours:Minutes:Seconds
        g_mins, g_secs = divmod(int(global_eta_seconds), 60)
        g_hours, g_mins = divmod(g_mins, 60)
        global_eta_display = f" | Script ETA: {g_hours}h {g_mins}m {g_secs}s"
    else:
        global_eta_display = " | Script ETA: Calculating..."
        
    print(f"\n--- Processing Player {player_index + 1}/{total_players}{global_eta_display} ---")
    
    # 1. Get their last Ranked games (Note: You had total_matches=2 here, adjust as needed!)
    recent_match_ids = get_player_match_history(target_player, total_matches=250)
    
    # 2. Parse the details for each game and store in a list
    all_match_records = []
    
    if recent_match_ids:
        total_matches = len(recent_match_ids)
        print(f"Parsing details for {total_matches} matches...")
        
        # Start the timer for this specific player's matches
        player_start_time = time.time()

        for index, m_id in enumerate(recent_match_ids):
            # Calculate the local ETA for the current player's games
            if index > 0:
                elapsed_time = time.time() - player_start_time
                avg_time_per_match = elapsed_time / index
                matches_left = total_matches - index
                eta_seconds = avg_time_per_match * matches_left
                
                # Format the remaining seconds into a readable minutes:seconds format
                mins, secs = divmod(int(eta_seconds), 60)
                eta_display = f" | Local ETA: {mins}m {secs}s"
            else:
                eta_display = " | Local ETA: Calculating..."

            print(f" Player {player_index + 1}/{total_players}{global_eta_display} -> Processing match {index + 1}/{total_matches} (ID: {m_id}){eta_display}")
            
            record = parse_match_data(m_id, target_player)
            if record:
                all_match_records.append(record)
            
            # Gentle sleep to respect rate limits
            time.sleep(1.2)
            
        # 3. Convert to a Pandas DataFrame for easy viewing and analysis
        df = pd.DataFrame(all_match_records)
        
        # Convert the milliseconds timestamps to readable Datetime objects
        if not df.empty and 'game_end_timestamp' in df.columns:
            df['game_end_datetime'] = pd.to_datetime(df['game_end_timestamp'], unit='ms')
        
        save_player_data(df, target_player)